# Preprocessing Technique: Feature Engineering — Feature Selection
**Member:** [GIMSARA H.S.M. / IT25103688]
**Technique:** Feature Engineering (Feature Selection half — dimension reduction is handled
separately by a teammate using PCA)
**Dataset:** Tourism Recommendation Dataset
**Target variable:** `satisfaction_level`



In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt

# Missing values are already handled before we select features.
df = pd.read_csv("output_after_missing_data.csv")
sat_order = ["Neutral", "Satisfied", "Very Satisfied"]

In [2]:
# Step 1: Check for LEAKAGE - does another column perfectly/near-perfectly map to the target
leak_check = pd.crosstab(df["satisfaction_level"], df["recommendation_level"])
print(leak_check)

recommendation_level  Highly Recommend  Neutral  Recommend
satisfaction_level                                        
Neutral                              0      241          0
Satisfied                            0        0      50518
Very Satisfied                   49241        0          0


In [3]:
# Step 2: Check rating (numeric) - is it also a leakage risk
print(df.groupby("satisfaction_level")["rating"].mean())


satisfaction_level
Neutral           3.295021
Satisfied         4.140473
Very Satisfied    4.736468
Name: rating, dtype: float64


In [4]:
## Filter Method 1: 
## Chi-Square Test: Checks the relationship between categorical features and the categorical target.
## A high Chi-Square value and low p-value indicate the feature is strongly related to the target and is useful for feature selection.
def chi_square_test(col):
    contingency = pd.crosstab(df[col], df["satisfaction_level"])
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    return chi2, p_value

candidate_cats = ["gender", "age_group", "season", "is_holiday", "is_group_tour",
                   "attraction_level", "attraction_category", "province", "source_province"]

chi2_results = pd.DataFrame(
    {col: chi_square_test(col) for col in candidate_cats},
    index=["chi2_statistic", "p_value"]
).T.sort_values("chi2_statistic", ascending=False)

chi2_results

,chi2_statistic,p_value
attraction_level,23446.205580,0.000000
attraction_category,16317.407977,0.000000
province,3015.586358,0.000000
source_province,78.435801,0.140464
age_group,8.823446,0.357408
is_holiday,5.835028,0.054068
gender,3.735630,0.154461
season,2.631491,0.853471
is_group_tour,0.223552,0.894244


In [5]:
## Filter Method 2: Correlation Coefficient (numeric features vs. ordinal target)
df["sat_ord"] = df["satisfaction_level"].map({"Neutral": 0, "Satisfied": 1, "Very Satisfied": 2})
numeric_candidates = ["age", "ticket_price", "visit_duration_hours", "spend_amount", "other_spend"]
corr_scores = df[numeric_candidates].corrwith(df["sat_ord"]).sort_values(key=abs, ascending=False)
corr_scores

ticket_price            0.138148
spend_amount            0.057485
other_spend             0.006335
age                    -0.004114
visit_duration_hours   -0.001450
dtype: float64

In [6]:
# Step 5: Final feature selection decision based on the filter test evidence above
selected_features = [
    "attraction_level",   
    "attraction_category",  
    "province",               
    "ticket_price",
    "spend_amount",
]
excluded_leakage = ["recommendation_level", "rating", "tourist_id"]
excluded_weak = ["gender", "age", "age_group", "season", "is_holiday", "is_group_tour",
                  "visit_duration_hours", "other_spend", "source_province"]

print("Selected:", selected_features)
print("\nExcluded (leakage):", excluded_leakage)
print("\nExcluded (negligible signal per chi-square/correlation):", excluded_weak)

Selected: ['attraction_level', 'attraction_category', 'province', 'ticket_price', 'spend_amount']

Excluded (leakage): ['recommendation_level', 'rating', 'tourist_id']

Excluded (negligible signal per chi-square/correlation): ['gender', 'age', 'age_group', 'season', 'is_holiday', 'is_group_tour', 'visit_duration_hours', 'other_spend', 'source_province']


In [7]:
## Generating Output
## Build the final selected-feature dataset (keeping `satisfaction_level` as the target) and save it for the next person (Outlier Removal).
df_selected = df[selected_features + ["satisfaction_level"]].copy()
df_selected.to_csv("output_after_feature_selection.csv", index=False)
print("Saved output_after_feature_selection.csv - shape:", df_selected.shape)


Saved output_after_feature_selection.csv - shape: (100000, 6)
